# Burgers 1D Dataset Exploration

This notebook loads the committed `train.npz` and `test.npz` files for the viscous Burgers case, reports the snapshot counts, visualizes the first few saved training snapshots, and documents the parameter values used for every training trajectory and the held-out testing trajectory.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'train.npz').exists():
    ROOT = Path('data/burgers_1d').resolve()
TRAIN_PATH = ROOT / 'train.npz'
TEST_PATH = ROOT / 'test.npz'

train = np.load(TRAIN_PATH, allow_pickle=False)
test = np.load(TEST_PATH, allow_pickle=False)
train_meta = json.loads(str(train['metadata_json']))
test_meta = json.loads(str(test['metadata_json']))

PARAMETER_DESCRIPTIONS = {
    'viscosity': 'Diffusion strength in the viscous Burgers equation.',
    'mean_level': 'Conserved spatial mean that the periodic solution relaxes toward.',
    'initial_amplitude': 'Amplitude used for the smooth periodic initial condition.',
    'initial_condition_seed': 'Deterministic seed used to construct the random Fourier-like initial condition.'
}

def split_summary(dataset, metadata):
    num_trajectories = int(dataset['parameter_matrix'].shape[0])
    num_snapshots = int(dataset['states'].shape[0])
    snapshots_per_trajectory = dataset['trajectory_lengths']
    print(f"Split: {dataset['split_name'].item()}")
    print(f"  trajectories: {num_trajectories}")
    print(f"  total snapshots: {num_snapshots}")
    print(f"  snapshots per trajectory: {snapshots_per_trajectory.tolist()[:5]}{' ...' if len(snapshots_per_trajectory) > 5 else ''}")
    print(f"  state shape per snapshot: {dataset['states'].shape[1:]}")
    print(f"  saved time window: [{dataset['times'].min():.4f}, {dataset['times'].max():.4f}]")
    print(f"  steady-time estimate range: [{dataset['steady_time_estimates'].min():.4f}, {dataset['steady_time_estimates'].max():.4f}]")
    print(f"  equation: {metadata['equation']}")

def trajectory_block(dataset, trajectory_index):
    start = int(dataset['trajectory_offsets'][trajectory_index])
    stop = int(dataset['trajectory_offsets'][trajectory_index + 1])
    return dataset['states'][start:stop], dataset['times'][start:stop]

split_summary(train, train_meta)
print()
split_summary(test, test_meta)


## Parameter Documentation

The stored parameter columns are deterministic and aligned with the `parameter_matrix` rows in both splits. Each training row corresponds to one contiguous time block in `states`, and the testing archive contains the single held-out trajectory.

In [ ]:
parameter_names = train['parameter_names'].tolist()
print('Parameter names and meanings:')
for name in parameter_names:
    print(f"- {name}: {PARAMETER_DESCRIPTIONS.get(name, 'No description available.')}")

print('\nTraining trajectory parameter table:')
header = ['traj_id', 't_start', 't_end', *parameter_names]
print(' | '.join(header))
for traj_id, params in enumerate(train['parameter_matrix']):
    states_i, times_i = trajectory_block(train, traj_id)
    row = [traj_id, f"{times_i[0]:.4f}", f"{times_i[-1]:.4f}", *[f"{value:.6g}" for value in params]]
    print(' | '.join(map(str, row)))

print('\nHeld-out testing trajectory parameters:')
test_states_0, test_times_0 = trajectory_block(test, 0)
print('time window:', f"[{test_times_0[0]:.4f}, {test_times_0[-1]:.4f}]")
for name, value in zip(parameter_names, test['parameter_matrix'][0]):
    print(f"- {name}: {value:.6g}")


## Exploratory Plots

The next plots focus on the first training trajectory. The first figure shows a few early saved timesteps, and the second tracks coarse summary statistics across every saved training snapshot.

In [ ]:
x = train['x']
states0, times0 = trajectory_block(train, 0)
plot_indices = np.linspace(0, len(times0) - 1, min(4, len(times0)), dtype=int)

fig, ax = plt.subplots(figsize=(10, 5))
for idx in plot_indices:
    ax.plot(x, states0[idx], label=f"t = {times0[idx]:.4f}")
ax.set_title('First training trajectory: selected saved timesteps')
ax.set_xlabel('x')
ax.set_ylabel('u(x, t)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

train_states = train['states']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(train['times'], train_states.mean(axis=1))
axes[0].set_title('Mean over all training snapshots')
axes[0].set_xlabel('time')
axes[0].grid(True, alpha=0.25)
axes[1].plot(train['times'], train_states.min(axis=1))
axes[1].set_title('Minimum over all training snapshots')
axes[1].set_xlabel('time')
axes[1].grid(True, alpha=0.25)
axes[2].plot(train['times'], train_states.max(axis=1))
axes[2].set_title('Maximum over all training snapshots')
axes[2].set_xlabel('time')
axes[2].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(train['parameter_matrix'][:, 0], bins=12, alpha=0.8, edgecolor='black')
plt.title('Training distribution of viscosity values')
plt.xlabel('viscosity')
plt.ylabel('count')
plt.grid(True, alpha=0.25)
plt.show()
